# Cohort analysis: do the recurring groups meet week after week?

**Pipeline:** `episodes_v1` → transactions → frequent itemsets → association rules → **cohort analysis (this notebook)** → `outputs/tables/recurring_cohorts.csv`.

**Question:** support only says that a group shared at least 200 episodes, not when. For each cohort, this notebook measures when its episodes happen, how large they are and how many separate meetings they form.

**Cohort:** a closed frequent itemset of at least two students at minsup 200.

**Time conventions:** day = `start_time // 86400` (day 0 is a Sunday), week = day // 7, weekday = day % 7 (0 = Sunday). Clock hours assume that timestamp 0 is midnight, which the dataset does not state, so hours and night shares are descriptive only.

**Limit:** recurrence is still co-presence, not a social tie. Whether a cohort recurs more than the schedule explains is tested by the null models, which start from the CSV.

## Step 1. Load the episodes

Settings, the episodes sorted by `episode_id`, one transaction per episode and the True/False table for mlxtend.

In [ ]:
from itertools import combinations
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from mlxtend.frequent_patterns import apriori as mlx_apriori

PROJECT_ROOT = Path.cwd().parent                   # notebooks/ is one level below the project root
PROCESSED = PROJECT_ROOT / "data" / "processed"
TABLES = PROJECT_ROOT / "outputs" / "tables"

sys.path.append(str(PROJECT_ROOT))                 # so "from src..." works inside notebooks/
from src.config import load_config
from src.features.transactions import build_transactions, support_count, to_one_hot

config = load_config(PROJECT_ROOT / "config" / "config.yaml")
SLOT = config["time_parameters"]["slot_duration_sec"]    # 300 seconds
MINSUP = 200                                             # count of episodes, the chosen minimum support

episodes = pd.read_parquet(PROCESSED / "episodes_v1.parquet").sort_values("episode_id").reset_index(drop=True)
transactions = build_transactions(episodes)        # same order as the rows of `episodes`
n_episodes = len(transactions)
onehot = to_one_hot(transactions)                  # True where the student is in the episode
print(n_episodes, "episodes,", onehot.shape[1], "students")

## Step 2. Frequent itemsets at minsup 200

As in `association_rules.ipynb`: mlxtend `apriori` with `(MINSUP − 0.5) / N`, and the supports converted back to counts.

In [ ]:
share = (MINSUP - 0.5) / n_episodes
frequent = mlx_apriori(onehot, min_support=share, use_colnames=True, low_memory=True)

supports = {}                                      # {sorted tuple of student ids: support count}
for itemset, support in zip(frequent["itemsets"], frequent["support"]):
    key = tuple(sorted(int(item) for item in itemset))
    supports[key] = round(support * n_episodes)    # fraction back to a count

print(len(supports), "frequent itemsets at minsup", MINSUP)

## Step 3. Cohorts

A group is not closed if a group with one more student has the same support. The closed groups of at least two students are the cohorts. Expected: 735 closed itemsets and 343 cohorts.

In [ ]:
not_closed = set()
for itemset, support in supports.items():
    if len(itemset) < 2:
        continue
    for smaller in combinations(itemset, len(itemset) - 1):    # the group without one student
        if supports[smaller] == support:                       # same support: the smaller group is not closed
            not_closed.add(smaller)

closed = {}
for itemset, support in supports.items():
    if itemset not in not_closed:
        closed[itemset] = support

cohorts = {}                                       # closed groups of two or more students
for itemset, support in closed.items():
    if len(itemset) >= 2:
        cohorts[itemset] = support

print(len(closed), "closed itemsets,", len(cohorts), "cohorts. Not closed:", sorted(not_closed))
assert len(closed) == 735 and len(cohorts) == 343

## Step 4. Episode attributes

Day, week, weekday, number of students, a night flag (`night_share` ≥ 0.5) and a large flag (at least the 95th percentile of students per episode). Large episodes matter because one episode of 40 students contains 780 pairs.

In [ ]:
episodes["day"] = episodes["start_time"] // 86400          # day 0 .. 27
episodes["week"] = episodes["day"] // 7                    # week 0 .. 3
episodes["weekday"] = episodes["day"] % 7                  # 0 = Sunday
episodes["n_students"] = [len(t) for t in transactions]
episodes["is_night"] = episodes["night_share"] >= 0.5      # mostly between 00:00 and 07:00

large_threshold = episodes["n_students"].quantile(0.95)
episodes["is_large"] = episodes["n_students"] >= large_threshold

print("large episode = at least", large_threshold, "students;", int(episodes["is_large"].sum()), "large episodes,",
      int(episodes["is_night"].sum()), "night episodes")

## Step 5. Helper functions

- `count_sessions`: the number of separate sessions; episodes less than one hour apart are merged.
- `lift_range`: the lowest and highest lift over all splits of a cohort into $A$ and $B$.

In [ ]:
def count_sessions(rows, max_gap_sec):
    rows = rows.sort_values("start_time")
    sessions = 0
    last_end = None
    for start, end in zip(rows["start_time"], rows["end_time"] + SLOT):    # end_time is the start of the last slot
        if last_end is None or start - last_end > max_gap_sec:
            sessions += 1                                                  # a gap: a new session starts
            last_end = end
        else:
            last_end = max(last_end, end)                                  # same session goes on
    return sessions


def lift_range(cohort, support, supports, n_total):
    lifts = []
    for size in range(1, len(cohort)):
        for antecedent in combinations(cohort, size):
            consequent = tuple(item for item in cohort if item not in antecedent)
            lifts.append(n_total * support / (supports[antecedent] * supports[consequent]))
    return min(lifts), max(lifts)

## Step 6. One row per cohort

For each cohort: its episodes (those containing all its students), a recount of the support and every measure. The result, `table`, is sorted by support.

In [ ]:
rows = []
for cohort, support in cohorts.items():
    mask = onehot[list(cohort)].all(axis=1)                # episodes that contain every student of the cohort
    mine = episodes[mask]
    assert len(mine) == support                            # recount of the support
    low, high = lift_range(cohort, support, supports, n_episodes)
    rows.append({
        "members": " ".join(str(student) for student in cohort),
        "size": len(cohort),
        "support": support,
        "days_seen": mine["day"].nunique(),
        "weeks_seen": mine["week"].nunique(),
        "support_first_half": int((mine["day"] < 14).sum()),
        "support_second_half": int((mine["day"] >= 14).sum()),
        "top_weekday_share": mine["weekday"].value_counts().iloc[0] / support,
        "top_hour_share": mine["start_hour"].value_counts().iloc[0] / support,
        "night_episode_share": mine["is_night"].mean(),
        "night_share_mean": mine["night_share"].mean(),
        "mean_episode_students": mine["n_students"].mean(),
        "large_episode_share": mine["is_large"].mean(),
        "support_without_large": int((~mine["is_large"]).sum()),
        "survives_without_large": int((~mine["is_large"]).sum()) >= MINSUP,
        "sessions_1h": count_sessions(mine, 3600),
        "min_lift": low,
        "max_lift": high,
    })

table = pd.DataFrame(rows).sort_values(["support", "members"], ascending=[False, True]).reset_index(drop=True)
print(len(table), "cohorts; by size:", table["size"].value_counts().sort_index().to_dict(),
      "| support from", table["support"].min(), "to", table["support"].max(), ", median", table["support"].median())
table.head(10).round(3)

## Step 7. Second support check

Every support is recounted with `support_count` from `transactions.py`.

In [ ]:
for cohort, support in cohorts.items():
    assert support_count(transactions, cohort) == support
print("all", len(cohorts), "supports confirmed")

## Step 8. Do the cohorts recur week after week?

Weeks seen (of 4), the cohorts that miss a week, days seen (of 28) and the split of the support between the first and second half (days 0–13 and 14–27).

In [ ]:
print("Cohorts by number of weeks (of 4) in which they appear")
display(table["weeks_seen"].value_counts().sort_index().to_frame("cohorts"))
display(table[table["weeks_seen"] < 4][["members", "support", "days_seen", "weeks_seen", "support_first_half", "support_second_half"]])

print("Days seen (of 28), minimum / quartiles / maximum:", table["days_seen"].quantile([0, 0.25, 0.5, 0.75, 1]).tolist())
both_halves = (table["support_first_half"] > 0) & (table["support_second_half"] > 0)
print("In both halves of the 28 days:", int(both_halves.sum()), "of", len(table))
first_share = table["support_first_half"] / table["support"]
print("Share of the support in the first half, minimum / quartiles / maximum:", first_share.quantile([0, 0.25, 0.5, 0.75, 1]).round(2).tolist())

### Plots

Left: support against days seen. Right: support in the first half against the second half; the grey line marks equal support.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].scatter(table["support"], table["days_seen"], s=14, alpha=0.5, color="#2a78d6")
axes[0].set_xlabel("support (episodes)")
axes[0].set_ylabel("days seen (of 28)")
axes[0].set_title("Support against days seen")
axes[0].grid(alpha=0.3)

axes[1].scatter(table["support_first_half"], table["support_second_half"], s=14, alpha=0.5, color="#2a78d6")
top = max(table["support_first_half"].max(), table["support_second_half"].max())
axes[1].plot([0, top], [0, top], color="#898781", lw=1)    # same support in both halves
axes[1].set_xlabel("support in days 0-13")
axes[1].set_ylabel("support in days 14-27")
axes[1].set_title("First half against second half")
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Step 9. Regularity and night

The share of a cohort's episodes on its most common weekday (baseline 1/7) and start hour (baseline 1/24), and the number of cohorts that are mostly at night. The largest of several shares exceeds its baseline by chance, so these are rough indicators.

In [ ]:
print("top weekday share, minimum / quartiles / maximum:", table["top_weekday_share"].quantile([0, 0.25, 0.5, 0.75, 1]).round(2).tolist())
print("top hour share, minimum / quartiles / maximum:", table["top_hour_share"].quantile([0, 0.25, 0.5, 0.75, 1]).round(2).tolist())
print("cohorts with at least half of their episodes at night:", int((table["night_episode_share"] >= 0.5).sum()), "of", len(table))
table.head(10)[["members", "size", "support", "days_seen", "weeks_seen", "night_episode_share", "top_hour_share"]].round(3)

## Step 10. Large episodes

How many cohorts depend on the 5% largest episodes, and how many fall below minsup 200 without them.

In [ ]:
print("cohorts with at least half of their episodes large:", int((table["large_episode_share"] >= 0.5).sum()), "of", len(table))
below = table["support_without_large"] < MINSUP
print("cohorts below minsup", MINSUP, "without the large episodes:", int(below.sum()), "of", len(table))
print("share of a cohort's episodes that are large, minimum / quartiles / maximum:",
      table["large_episode_share"].quantile([0, 0.25, 0.5, 0.75, 1]).round(2).tolist())

## Step 11. Sessions

Support counts episodes, not meetings, and the linking can split one meeting into several episodes. Episodes less than one hour apart are merged into sessions; the one-hour gap is a diagnostic choice.

In [ ]:
print("sessions per cohort, median:", table["sessions_1h"].median(), "| maximum:", table["sessions_1h"].max())
print("episodes per session, median:", round((table["support"] / table["sessions_1h"]).median(), 1))
print("sessions per day seen, median:", round((table["sessions_1h"] / table["days_seen"]).median(), 2))

## Step 12. Cohorts that do not depend on large episodes

`survives_without_large` marks the cohorts that keep at least 200 episodes without the large ones. They are the closest to small recurring groups.

In [ ]:
robust = table[table["survives_without_large"]]
others = table[~table["survives_without_large"]]
print(len(robust), "cohorts survive; by size:", robust["size"].value_counts().sort_index().to_dict())
print("weeks seen:", robust["weeks_seen"].value_counts().sort_index().to_dict(), "| mostly at night:", int((robust["night_episode_share"] >= 0.5).sum()))
print("median students per episode:", round(robust["mean_episode_students"].median(), 1), "for these,",
      round(others["mean_episode_students"].median(), 1), "for the others")
robust.head(10)[["members", "size", "support", "support_without_large", "days_seen", "night_episode_share", "sessions_1h"]].round(3)

## Step 13. Write the CSV

One row per cohort, plus the settings that produced it (episodes version, minsup, θ, gap, RSSI threshold). No cohort is removed; `survives_without_large` allows filtering later.

In [ ]:
out = table.copy()
out["episodes_version"] = "episodes_v1"
out["minsup"] = MINSUP
out["jaccard_threshold"] = config["episode_parameters"]["jaccard_threshold"]
out["max_gap_slots"] = config["episode_parameters"]["max_gap_slots"]
out["rssi_threshold"] = config["bluetooth_parameters"]["rssi_threshold"]
for column in ["top_weekday_share", "top_hour_share", "night_episode_share", "night_share_mean",
               "mean_episode_students", "large_episode_share", "min_lift", "max_lift"]:
    out[column] = out[column].round(4)

TABLES.mkdir(parents=True, exist_ok=True)
out.to_csv(TABLES / "recurring_cohorts.csv", index=False)

check = pd.read_csv(TABLES / "recurring_cohorts.csv")
assert len(check) == 343 and list(check.columns) == list(out.columns)
print("wrote", TABLES / "recurring_cohorts.csv", check.shape)

### Findings (`episodes_v1`, minsup 200)

- **343 cohorts:** 181 pairs, 90 trios, 55 groups of four, 16 of five and one of six; support from 200 to 523 (median 226).
- **Recurrence:** 340 cohorts appear in all four weeks and the other three in three weeks; all appear in both halves. The median cohort is seen on 19 of 28 days, with its support split about evenly between the halves.
- **Episodes are not meetings:** a cohort forms a median of 20 sessions (at most 75), about 11.6 episodes per session and one session per day seen.
- **Large episodes:** 241 cohorts have at least half of their episodes among the 5% largest (26 students or more), and 296 fall below minsup without them. The 47 that remain (41 pairs, 6 trios) meet in episodes of about 5 students, against 41 for the others.
- **Regularity:** the median share on the most common weekday is 0.30 (baseline 0.14), and on the most common start hour 0.22 (baseline 0.04).
- **Night:** 19 cohorts are mostly at night, 18 of them among the 47. The four cohorts with the highest support (students 47, 448 and 449) are about half at night, with about 5 students per episode; this fits shared housing, which the data cannot confirm.
- **Answer:** almost every cohort recurs in every week, but most are co-presence in large episodes. Whether any cohort recurs more than the schedule explains is the null-model test.
- **Limits:** weeks are 7-day blocks from timestamp 0; hours assume that timestamp 0 is midnight; the large-episode threshold (95th percentile) and the one-hour session gap are choices; all results are for RSSI −90 dBm, θ 0.7 and g 1.